# Fine-tuning speed & memory benchmark (≈ 15–25 min on A100)

Runs 40 real training steps per model (no checkpoints, no predictions, nothing written except
`My Drive/zuzai-hw/bench/<model>.json`) and prints seconds/step, peak GPU memory and the estimated time per fold.
**Runtime → Change runtime type → G4 or A100 → Runtime → Run all**, then paste the `BENCH` lines to Claude.

In [ ]:
# (model, extra flags), all with the same effective batch of 16. Without gradient checkpointing an 8B model needs
# far more activation memory, so on a 40 GB A100 only small micro-batches may fit; an "out of memory" line just
# means that setting is not possible on this GPU.
RUNS = [
    ("qwen3-8b-lora", ""),                                   # as benchmarked on A100: gc on, 8 × 2 → 1.35 s/step
    ("qwen3-8b-lora", "--no-gc --batch-size 4 --accum 4"),   # no gradient checkpointing, small micro-batch
    ("qwen3-8b-lora", "--no-gc --batch-size 8 --accum 2"),   # no gc, original micro-batch (needs ≈ 50 GB)
    ("qwen3-8b-lora", "--no-gc --batch-size 16 --accum 1"),  # no gc, one batch of 16 (needs ≈ 80 GB)
]
STEPS = 40
REPO = "https://github.com/ShayHummel/work-type-labels.git"
BRANCH = "solution"
OUT = "/content/drive/MyDrive/zuzai-hw"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
if not os.path.exists("/content/work-type-labels"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/work-type-labels"], check=True)
else:
    subprocess.run(["git", "-C", "/content/work-type-labels", "pull", "-q"], check=True)
%cd /content/work-type-labels
!git log --oneline -1
!pip install -q "transformers==5.18.0" "accelerate==1.15.0" "peft==0.21.2" "sentencepiece==0.2.2" "protobuf"
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
for m, flags in RUNS:
    print(f"===== {m} {flags} =====", flush=True)
    !python src/finetune.py --model {m} --folds 0 --bench --max-steps {STEPS} {flags} --out {OUT} --ckpt-root /content/ckpt_bench 2>&1 | grep -E "BENCH|Error|error:|out of memory"

In [ ]:
!grep -h BENCH {OUT}/logs/ft_*.log | tail -n 10